# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt
import rasterio
import numpy as np

import os

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

save_filtered_attributes = True

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
    print("Geometries cleaned")
    
    if save_filtered_attributes:
        if row['save_format'] == 'parquet':
            if not os.path.exists(f'{output_step2_path}/gpkg_attributs'):
                os.makedirs(f'{output_step2_path}/gpkg_attributs')
            gdf.to_parquet(f"{output_step2_path}/parquet_attributs/{attribute}.parquet")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        elif row['save_format'] == 'gpkg':
            if not os.path.exists(f'{output_step2_path}/parquet_attributs'):
                os.makedirs(f'{output_step2_path}/parquet_attributs')
            gdf.to_file(f"{output_step2_path}/gpkg_attributs/{attribute}.gpkg", driver="GPKG")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        else:
            if not os.path.exists(f'{output_step2_path}/csv_attributs'):
                os.makedirs(f'{output_step2_path}/csv_attributs')
            gdf.to_csv(f"{output_step2_path}/csv_attributs/{attribute}.csv", index=False)
            print(f"Warning: Unknown save format {row['save_format']} for attribute {attribute}. Data saved as csv.")
    else:
        print("Note : Save option is disabled.")

#### Import des attributs 

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

In [4]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,impact_attribut,file_name,geometry_type,method,how,value_column,buffer_size,filter_column,filter_values,crs,save_format
0,Sécurité,accident,accident,True,accident,0.3,defavorable,OTC_ACCIDENTS-SHP/OTC_ACCIDENTS.shp,point,A,count,NaN,10,filtered,1,2056,parquet
1,Sécurité,traffic,zone_apaisee,True,vitesse,0.5,favorable,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
2,Sécurité,traffic,zone_pietonne,True,vitesse,1.0,favorable,OTC_ZONE_MODERATION_TRAFIC-SHP/OTC_ZONE_MODERA...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
3,Sécurité,traffic,vitesse,True,vitesse,0.6,defavorable,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
4,Infrastructure,stationnement_genant,stationnement_genant,True,stationnement_genant,0.3,defavorable,SHP_FDP/FDP_STATIONNEMENT_GENANT_PIETON_CONTRA...,point,A,count,NaN,10,filtered,1,2056,parquet
5,Infrastructure,connectivite,connectivite,True,connectivite,0.7,favorable,NaN,line,A,sum,conn_branching_in_buffer,10,filtered,1,2056,parquet
6,Infrastructure,largeur_trottoir,largeur_trottoir,True,network_couche_OCT.shp,0.5,favorable,RP_final.shp,line,A,count,NaN,10,filtered,1,2056,parquet
7,Infrastructure,topographie,topographie,True,network_couche_OCT.shp,0.4,defavorable,RP_final.shp,line,A,sum,Pente,10,filtered,1,2056,parquet
8,Attractivité,eau,eau,True,eau,0.3,favorable,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,count,NaN,10,filtered,1,2056,parquet
9,Attractivité,proximite,rez_actif,True,rez_actif,0.5,favorable,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,count,NaN,10,filtered,1,2056,parquet


**Attribut Accidents**

In [5]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ANNEE > 2020, 'filtered'] = 1
gdf.loc[gdf.CONSEQ.isin(['Avec blessés graves', 'Avec tués']), 'filtered'] = 1
gdf.loc[gdf.VELOS == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_25 == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_45 == 1, 'filtered'] = 1
gdf.loc[gdf.PIETONS == 1, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: accident
Filters applied
Proportion of features accident kept after filtering:
filtered
0    0.582074
1    0.417926
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: accident in format: parquet


**Attribut Arbres isolés** (groupe Végétation)

In [6]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'arbre_isole'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: arbre_isole
Filters applied
Proportion of features arbre_isole kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: arbre_isole in format: parquet


**Attribut Espaces verts** (groupe Végétation)

In [7]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espace_vert'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.REVETEMENT.isin(['Arbustes', 'Terre', 'Gazon','Grille gazon', 'Prairie', 'Plates-bandes']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espace_vert
Filters applied
Proportion of features espace_vert kept after filtering:
filtered
0    0.642584
1    0.357416
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: espace_vert in format: parquet


**Attribut Vitesse** (groupe Traffic)

In [8]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE == 0, 'filtered'] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: vitesse
Filters applied
Proportion of features vitesse kept after filtering:
filtered
1    0.674852
0    0.325148
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: vitesse in format: parquet


**Attribut Zone pietonne** (groupe Traffic)

In [9]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_pietonne'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE.isin(['Zone piétonne', 'Zone de rencontre (20)', ]), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Filters applied
Proportion of features zone_pietonne kept after filtering:
filtered
0    0.536313
1    0.463687
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: zone_pietonne in format: parquet


/Users/Helo/miniconda3/envs/actionsituee/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(


**Attribut Zone apaisée** (groupe Traffic)

In [10]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_LIMIT.isin(['Zone 30 km/h', 'Prescription 30 km/h', 'Prescription 20 km/h']), 'filtered'] = 1 

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_apaisee
Filters applied
Proportion of features zone_apaisee kept after filtering:
filtered
0    0.821338
1    0.178662
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: zone_apaisee in format: parquet


**Eau**

In [12]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eau'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ETAT.isin(['A ciel ouvert']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Filters applied
Proportion of features eau kept after filtering:
filtered
1    0.615236
0    0.384764
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: eau in format: parquet


**Rez Actifs**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#gdf.loc[gdf['BRANCHE'].str.contains('commerce de détail|détail|écoles|commerces|supermarchés|restaurants|banques|enseignement', case=False, na=False), 'filtered'] = 1
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: rez_actif
Data initialized
Processing attribute: rez_actif
Filters applied
Proportion of features rez_actif kept after filtering:
filtered
0    0.86085
1    0.13915
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: rez_actif in format: parquet


**Attribut Bruit**

In [8]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bruit'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Simplify geometries
simplify_tolerance = 1.0  # in meters
print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
print("Simplification done.")
###--------------------

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: bruit
Simplifying geometries with tolerance = 1.0 m ...
Simplification done.
Filters applied
Proportion of features bruit kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: bruit in format: parquet


**Attribut Proximité TP**

In [20]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tp'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: tp
Filters applied
Proportion of features tp kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: tp in format: parquet


**Attribut Stationnement Genant**

In [21]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_genant'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: stationnement_genant
Filters applied
Proportion of features stationnement_genant kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: stationnement_genant in format: parquet


**Attribut Proximité Aménités**

In [22]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: amenite
Filters applied
Proportion of features amenite kept after filtering:
filtered
0    0.86085
1    0.13915
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: amenite in format: parquet


**Attribut espaces ouverts**

In [23]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espaces_ouverts'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE == 'ESPACE PUBLIC', 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espaces_ouverts
Filters applied
Proportion of features espaces_ouverts kept after filtering:
filtered
0    0.589171
1    0.410829
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: espaces_ouverts in format: parquet


**Attribut Confort thermique**

In [ ]:
# Initialize

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Sample every 10th point
    rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
print(f"Saving {attribute}: can take up to 3mn")
save(save_filtered_attributes, row, gdf, attribute)

Processing attribute: temperature

Value counts (first 10 most common values):
temperature
23.399401    32014
23.399500     7482
23.403999      492
23.403700      480
23.404100      455
23.403900      433
23.403799      432
23.401600      409
23.402599      398
23.403601      387
Name: count, dtype: int64

Descriptive statistics:
count    847553.000000
mean         27.970154
std           3.923547
min          17.551901
25%          24.603500
50%          29.024401
75%          31.629601
max          34.583199
Name: temperature, dtype: float64
Saving temperature: ca take up to 3mn
Geometries cleaned
Filtered data saved for attribute: temperature in format: parquet


In [6]:
temp_parquet = gpd.read_parquet('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/parquet_attributs/temperature.parquet')
temp_parquet.to_file('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs/temperature.gpkg', driver="GPKG")


**Attribut Largeur trottoir**

In [26]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'largeur_trottoir'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.Largeur.isin(['Très large', 'Large','Moyen']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: largeur_trottoir
Filters applied
Proportion of features largeur_trottoir kept after filtering:
filtered
0    0.768267
1    0.231733
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: largeur_trottoir in format: parquet


**Attribut Topographie**

In [27]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'topographie'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: topographie
Filters applied
Proportion of features topographie kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: topographie in format: parquet


**Attribut Canopée**

In [7]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'canopee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Simplify geometries
simplify_tolerance = 1.0  # in meters
print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
print("Simplification done.")
###--------------------

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: canopee
Simplifying geometries with tolerance = 1.0 m ...
Simplification done.
Filters applied
Proportion of features canopee kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: canopee in format: parquet


/var/folders/xd/q3z8hkl97ss7mdr3m6y7rn340000gn/T/ipykernel_74689/3705390361.py:42: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  orig_area = gdf.geometry.area.sum()


Area difference: 0.31%


/var/folders/xd/q3z8hkl97ss7mdr3m6y7rn340000gn/T/ipykernel_74689/3705390361.py:43: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  simpl_area = gdf.geometry.simplify(1, preserve_topology=True).area.sum()


In [17]:
temp_parquet = gpd.read_parquet('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/parquet_attributs/canopee.parquet')
temp_parquet.to_file('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs/canopee.gpkg', driver="GPKG")